In [ ]:
from langchain_community.document_loaders import WebBaseLoader
import bs4
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Only keep post title, headers, and content from the full HTML.
bs4_strainer = bs4.SoupStrainer(class_=("post-title", "post-header", "post-content"))
# 加载器
loader = WebBaseLoader(
    web_paths=["https://lilianweng.github.io/posts/2023-06-23-agent/"],
    bs_kwargs={"parse_only": bs4_strainer},
)
# 加载文档
docs = loader.load()
print(len(docs))
print(f"Total characters: {len(docs[0].page_content)}")

# 打印前500字符
if len(docs[0].page_content) > 500 :
    print(docs[0].page_content[:500])

In [ ]:
# 创建文本拆分器，1000字符/块，重叠200字符并设置开始索引
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True
)
# 分割文档
all_splits = text_splitter.split_documents(docs)
print(len(all_splits))


In [ ]:
# 将分块数据使用嵌入模型转换为向量数据存醋
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="nomic-embed-text")
vectorstore = Chroma(embedding_function=embeddings)
document_ids = vectorstore.add_documents(documents=all_splits)
print(document_ids[:3])

In [ ]:
# 使用向量存储构建检索器，并设置搜索类型和关键字参数
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 6})
# 检索文档关联内容
retrieved_docs = retriever.invoke("What are the approaches to Task Decomposition?")
print(len(retrieved_docs))
print(retrieved_docs[0].page_content)

In [ ]:
import requests
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os

load_dotenv()
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
if not deepseek_api_key:
    raise ValueError("DEEPSEEK_API_KEY 环境变量未设置")

base_url = "https://api.deepseek.com/v1"

def list_models(api_key):
    headers = {"Authorization": f"Bearer {api_key}"}
    response = requests.get("https://api.deepseek.com/v1/models", headers=headers)
    return response.json()

# 使用
models = list_models(os.getenv("DEEPSEEK_API_KEY"))
print("可用模型:", [model["id"] for model in models["data"]])

In [ ]:
model = ChatOpenAI(
    base_url=base_url,
    api_key=deepseek_api_key,
    model="deepseek-chat",
)
# model.invoke("你是谁？").content

In [ ]:
from langchain.chat_models import init_chat_model
llm = init_chat_model(model="llama3", model_provider="ollama")
llm.invoke([("human","你是谁？"), {"role": "system", "content": "使用简体中文回答"}]).content

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model = "llama3",
    temperature = 0.8,
    num_predict = 256,
    # other params ...
)
messages = [ ("system", "使用中文回答"), ("human", "你是谁？")]
for chunk in llm.stream(messages):
    print(chunk.text(), end="")

In [ ]:
# 使用LangChain hud的 RAG提示符
from langchain import hub

prompt = hub.pull("rlm/rag-prompt")
example_msgs = prompt.invoke({
    "context": "filler context",
    "question": "filler question",
}).to_messages()
print(example_msgs)

In [ ]:
if len(example_msgs) > 0:
    print(example_msgs[0].content)

In [ ]:
# Runnable协议的链式调用
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 链式调用，存储器-格式化文档函数
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

for chunk in rag_chain.stream("What is Task Decomposition? 并将其翻译成中文"):
    print(chunk, end="", flush=True)

In [ ]:
# LangChain中的内置链
from langchain.chains.retrieval import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)
prompt = ChatPromptTemplate.from_messages([("system", system_prompt),("human", "{input}"),])

question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)
response = rag_chain.invoke({"input": "What is Task Decomposition? 并翻译成中文"})
print(response["answer"])
print("--------------------------------")
# 向用户展示用于生成答案的来源
for document in response["context"]:
    print(document)
    print()